In [22]:
import numpy as np
import gdspy
from autograd import numpy as npa

################################################### design parameter #########################
resolution = 50  # 해상도
Mask_thick = 25
design_region_x = round(0.4, 2)
design_region_y = round(7.0 + 2 * Mask_thick / resolution, 2)
design_region_z = round(7.0 + 2 * Mask_thick / resolution, 2)
design_region_resolution = int(resolution)
Nx = int(design_region_resolution * design_region_x) + 1
Ny = int(design_region_resolution * design_region_y) + 1
Nz = int(design_region_resolution * design_region_z) + 1
##############################################################################################

# Numpy 배열 생성 2D 배열을 만듭니다
structure_weight = np.loadtxt('lastdesign.txt')
structure_weight = structure_weight.reshape(Nx, Ny * Nz)[Nx - 1]
data = npa.rot90(structure_weight.reshape(Ny, Nz))  # ZxY

# Check if the cell 'TOP_RECT' and 'TOP_SMOOTH' exist and delete them if they do
if 'TOP_RECT' in gdspy.current_library.cells:
    del gdspy.current_library.cells['TOP_RECT']
if 'TOP_SMOOTH' in gdspy.current_library.cells:
    del gdspy.current_library.cells['TOP_SMOOTH']

# Create the new cells
rect_cell = gdspy.Cell('TOP_RECT')
smooth_cell = gdspy.Cell('TOP_SMOOTH')

# Rectangle dimensions
width = 1
height = 1

# Numpy 배열을 기반으로 GDS의 경로(Path) 생성
rectangles = []
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        if data[i, j] == 1:
            # 각 사각형의 좌표를 정밀하게 조정
            x0 = j * width
            y0 = -i * height
            x1 = (j + 1) * width
            y1 = -(i + 1) * height
            rectangle = gdspy.Rectangle((x0, y0), (x1, y1))
            rectangles.append(rectangle)

# Rectangle 방식으로 셀에 추가
if rectangles:
    for rectangle in rectangles:
        rect_cell.add(rectangle)

# Chaikin's corner-cutting algorithm with control over smoothing
def chaikin_smoothing(points, iterations=3, corner_ratio=0.25):
    """Smooth the polygon using Chaikin's algorithm with control over corner smoothing."""
    for _ in range(iterations):
        new_points = []
        for i in range(len(points) - 1):
            p1 = points[i]
            p2 = points[i + 1]
            # 각 두 점 사이에 corner_ratio, (1 - corner_ratio) 지점의 새로운 점을 추가
            new_points.append(((1 - corner_ratio) * p1[0] + corner_ratio * p2[0],
                               (1 - corner_ratio) * p1[1] + corner_ratio * p2[1]))
            new_points.append((corner_ratio * p1[0] + (1 - corner_ratio) * p2[0],
                               corner_ratio * p1[1] + (1 - corner_ratio) * p2[1]))
        new_points.append(points[-1])  # 마지막 점 추가
        points = new_points
    return points

# Smooth 방식으로 다각형 병합 및 처리
if rectangles:
    # 병합된 다각형을 생성
    # 직사각형을 살짝 확장하여 작은 간격을 메움
    expanded_rectangles = [gdspy.offset(rectangle, distance=0.05) for rectangle in rectangles]

    # 확장된 직사각형들을 병합
    merged_polygon = gdspy.boolean(expanded_rectangles, None, 'or', precision=1e-5, max_points=1000000)

    # 병합된 다각형이 존재할 때 처리
    if merged_polygon:
        polygons_to_save = []
        for polygon_boundary in merged_polygon.polygons:
            # Chaikin's smoothing 적용해 경계를 부드럽게 처리
            smoothed_points = chaikin_smoothing(polygon_boundary, iterations=10, corner_ratio=0.25)
            smoothed_polygon = gdspy.Polygon(smoothed_points)
            polygons_to_save.append(smoothed_polygon)
        
        # 다각형을 여러 개의 파일로 저장 (예시로 1000개씩 분할 저장)
        for idx, polygon in enumerate(polygons_to_save):
            if idx % 1000 == 0:
                # 새로운 파일에 추가
                gdspy.write_gds(f'output_smooth_{idx // 1000}.gds', cells=[smooth_cell])
            smooth_cell.add(polygon)

# Save the GDS files for both cases
gdspy.write_gds('output_rect.gds', cells=[rect_cell])
gdspy.write_gds('output_smooth.gds', cells=[smooth_cell])

# Optionally, save images of the cells as SVG.
rect_cell.write_svg('output_rect.svg')
smooth_cell.write_svg('output_smooth.svg')

# Display all cells using the internal viewer.
gdspy.LayoutViewer()



/tmp/ipykernel_1009908/1272079548.py:105: DeprecationWarning: [GDSPY] Use of the global library is deprecated.  Pass LayoutViewer a GdsLibrary instance.
  gdspy.LayoutViewer()


<gdspy.viewer.LayoutViewer object .!layoutviewer>